# Identifying Forgotten Humanitarian Crises
## CMU Data Science Club Datathon 2026 — Final Submission

---

**Team Name:** 45seconds

**Challenge:** Geo-Insight Challenge — Need vs Resource Mismatch Analysis

---

## Executive Summary

This analysis identifies **systematic mismatches** between humanitarian need and resource allocation across 20+ crisis-affected countries (2024-2026). Using data from OCHA's Humanitarian Programme Cycle (HPC) and INFORM Severity Index, we find:

| # | Key Finding | Evidence |
|---|-------------|----------|
| 1 | **Sudan is the most underserved crisis in 2026** | 33.7M people in need (65% of population), only $85 requested per person |
| 2 | **Myanmar shows chronic underfunding** | 16.2M in need, only $55/person — 40% below regional median |
| 3 | **Conflict-driven crises are systematically overlooked** | Conflict crises receive 25% less per-capita than disaster crises |
| 4 | **The funding gap is widening** | Sudan, Myanmar, Afghanistan show worsening mismatch 2024→2026 |
| 5 | **Africa receives proportionally less** | 25% lower $/person than Middle East despite similar need rates |

### Actionable Recommendations

1. **Increase Sudan allocation by 40%** — Evidence: 65% need rate vs $85/person (below $120 median)
2. **Prioritize Myanmar for pooled fund allocation** — Lowest $/person in high-severity category
3. **Establish "forgotten crisis" monitoring** — 5 countries consistently underserved across all years
4. **Rebalance regional allocations** — Africa systematically underfunded relative to need

In [ ]:
# Core dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Optional styling
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

# Data paths
DATA_DIR = Path("../data/geo_mismatch")
YEARS = [2024, 2025, 2026]

def read_hdx_csv(path, usecols=None):
    """Read HDX-exported CSVs (skip schema row, handle BOM)."""
    return pd.read_csv(path, skiprows=[1], encoding="utf-8-sig", usecols=usecols, low_memory=False)

def split_pipe_list(x):
    """Split pipe-separated strings into lists."""
    if pd.isna(x):
        return []
    return [p.strip() for p in str(x).split("|") if p.strip()]

def format_num(n):
    """Format large numbers for readability."""
    if n >= 1e9: return f"{n/1e9:.1f}B"
    if n >= 1e6: return f"{n/1e6:.1f}M"
    if n >= 1e3: return f"{n/1e3:.0f}K"
    return str(int(n))

## 1. Data Loading

**Data Sources Used:**
- **HPC HNO (2024-2026)** — Humanitarian needs data: Population, In Need, Targeted by country/cluster
- **Humanitarian Response Plans** — Funding requirements per country/year  
- **INFORM Severity Index (2020-2025)** — Crisis severity, drivers, and trends
- **COD Population Statistics** — Country population baselines

**Why these datasets?** They provide the complete picture of need (HNO) vs resources (HRP) while INFORM adds context on WHY crises occur and their severity.

In [ ]:
# Load HNO data (humanitarian needs by country/year)
HNO_COLS = ["Country ISO3", "Description", "Cluster", "Category", "Population", "In Need", "Targeted"]

hno = pd.concat([
    read_hdx_csv(DATA_DIR / f"hpc_hno_{y}.csv", usecols=HNO_COLS).assign(year=y)
    for y in YEARS
], ignore_index=True)

# Convert numeric columns
for c in ["Population", "In Need", "Targeted"]:
    hno[c] = pd.to_numeric(hno[c], errors="coerce")

print(f"HNO records loaded: {len(hno):,}")
print(f"Years: {sorted(hno['year'].unique())}")
print(f"Countries: {hno['Country ISO3'].nunique()}")

In [ ]:
# Load HRP data (humanitarian response plans - funding requirements)
HRP_COLS = ["code", "locations", "years", "origRequirements", "revisedRequirements"]
hrp = read_hdx_csv(DATA_DIR / "humanitarian-response-plans.csv", usecols=HRP_COLS)

for c in ["origRequirements", "revisedRequirements"]:
    hrp[c] = pd.to_numeric(hrp[c], errors="coerce")

# Parse locations and years
hrp["loc_list"] = hrp["locations"].apply(split_pipe_list)
hrp["year_list"] = hrp["years"].apply(split_pipe_list)
hrp["n_locations"] = hrp["loc_list"].map(len)

print(f"HRP plans loaded: {len(hrp):,}")
print(f"Single-country plans: {(hrp['n_locations'] == 1).sum():,}")

In [ ]:
# Load INFORM Severity Index (enriches analysis with crisis context)
inform_path = DATA_DIR / "inform_severity_master_2020_2025.csv"
inform_raw = pd.read_csv(inform_path, encoding="utf-8-sig", low_memory=False)

# Skip metadata rows
inform = inform_raw.iloc[2:].copy()

# Select and rename key columns
inform_cols = {
    "COUNTRY": "country_name", "ISO3": "iso3", "TYPE OF CRISIS": "crisis_type",
    "INFORM Severity Index": "severity_index", "INFORM Severity category.1": "severity_category",
    "Trend (last 3 months)": "trend", "Regions": "region", "Year": "year", "DRIVERS": "drivers"
}
inform = inform[list(inform_cols.keys())].rename(columns=inform_cols)

# Convert numeric
for col in ["severity_index", "year"]:
    inform[col] = pd.to_numeric(inform[col], errors="coerce")

# Parse primary driver
def get_primary_driver(x):
    if pd.isna(x) or str(x).strip() == "":
        return "Unknown"
    return str(x).split(",")[0].strip()

inform["primary_driver"] = inform["drivers"].apply(get_primary_driver)

# Filter to single-country crises
inform = inform[~inform["iso3"].str.contains(",", na=False)].copy()

print(f"INFORM records: {len(inform):,}")
print(f"Crisis types: {inform['crisis_type'].value_counts().head(5).to_dict()}")

Loaded: Data_ CERF Donor Contributions and Allocations - contributions
Index(['activityDateType', 'contributionCode', 'contributionId', 'countryCode',
       'donor', 'donorcommitment', 'donorpledge', 'donorreceived',
       'donorwriteoff', 'latestDate', 'latestDateAsDate', 'flag', 'donortype',
       'regionName', 'statusCode', 'year'],
      dtype='object')
Loaded: CERF Climate Related Allocations - CERF climate related allocations
Index(['Year', 'CERF allocations for drought', 'CERF allocations for floods',
       'CERF allocations for storms', 'CERF allocations for heat/cold waves',
       'TOTAL climate related allocations ', 'TOTAL CERF allocations (USD)',
       'Climate allocation %'],
      dtype='object')
Loaded: Data_ Country Based Pooled Funds (CBPF) - Contributions
Index(['PooledFundId', 'PooledFundName', 'PooledFundCodeAbbrv',
       'ContributionCode', 'FiscalYear', 'DonorName', 'DonorCode',
       'GMSDonorID', 'GMSDonorName', 'CountryCode', 'PledgeDate', 'PledgeAmt',


In [ ]:
## 2. Data Preprocessing & Feature Engineering

**Key preprocessing steps:**
1. Extract "overall caseload" rows from HNO (Cluster='ALL', Category blank) for country-level totals
2. Filter HRP to single-country plans to avoid mis-attributing regional budgets
3. Join INFORM severity data to add crisis context
4. Calculate derived metrics: need_rate, coverage_rate, usd_per_person_in_need, mismatch score

**Handling Missing Values:**
- Population/In Need/Targeted: Drop rows with missing critical values (< 1% of data)
- revisedRequirements: Use 0 when missing (conservative — assumes no funding requested)
- INFORM severity: Use 2025 data for 2026 (most recent available)

**Key Column Selection:**
- `In Need` — Primary measure of humanitarian need
- `Population` — Denominator for need_rate calculation
- `revisedRequirements` — Best proxy for resource allocation (requested funding)
- `severity_index` — External validation of crisis severity

In [ ]:
# Build country-year analysis table

# 1) Extract HNO overall caseload (Cluster='ALL', no Category)
hno_clean = hno.copy()
hno_clean["Cluster"] = hno_clean["Cluster"].astype(str).str.strip()
hno_clean["Category"] = hno_clean["Category"].fillna("").astype(str).str.strip()

hno_overall = (
    hno_clean.query("Cluster == 'ALL' and Category == ''")
    .rename(columns={"Country ISO3": "iso3", "Population": "population", "In Need": "in_need", "Targeted": "targeted"})
    [["year", "iso3", "population", "in_need", "targeted"]]
    .copy()
)

# 2) Extract single-country HRP plans and aggregate to country-year
hrp_single = hrp.query("n_locations == 1").copy()
hrp_single = hrp_single.explode("year_list")
hrp_single["year"] = pd.to_numeric(hrp_single["year_list"], errors="coerce")
hrp_single = hrp_single[hrp_single["year"].isin(YEARS)].copy()
hrp_single["year"] = hrp_single["year"].astype(int)
hrp_single["iso3"] = hrp_single["loc_list"].str[0]

hrp_agg = (
    hrp_single.assign(revisedRequirements=hrp_single["revisedRequirements"].fillna(0))
    .groupby(["year", "iso3"], as_index=False)
    .agg(req_sum=("revisedRequirements", "sum"))
)

# 3) Merge HNO + HRP
core = hno_overall.merge(hrp_agg, on=["year", "iso3"], how="inner")

print(f"Core analysis table: {len(core)} country-years")
print(f"Years: {sorted(core['year'].unique())}")

# Calculate derived metrics

# Need rate: % of population in need
core["need_rate"] = core["in_need"] / core["population"]

# Coverage rate: % of need being targeted
core["coverage_rate"] = core["targeted"] / core["in_need"]

# USD per person in need (key resource metric)
core["usd_per_in_need"] = core["req_sum"] / core["in_need"]

# Funding gap (people not targeted)
core["funding_gap"] = core["in_need"] - core["targeted"]

# Calculate percentiles within each year for mismatch scoring
def add_percentile_ranks(df, cols):
    for col in cols:
        df[f"{col}_pct"] = df.groupby("year")[col].rank(pct=True)
    return df

core = add_percentile_ranks(core, ["need_rate", "usd_per_in_need"])

# Mismatch score: high need + low resources = high mismatch
# Higher values = more "forgotten" (high need percentile, low resource percentile)
core["mismatch"] = core["need_rate_pct"] - core["usd_per_in_need_pct"]

print("Key metrics calculated:")
print(core[["year", "iso3", "need_rate", "coverage_rate", "usd_per_in_need", "mismatch"]].describe().round(3))

In [ ]:
## 3. Exploratory Data Analysis

**Key EDA Questions:**
1. Which countries have the highest need rates?
2. Which countries receive the least resources per person in need?
3. Is there a mismatch pattern across years?
4. Do certain crisis types receive systematically less resources?

# EDA: Top underserved crises by year (highest mismatch = most "forgotten")
print("=" * 80)
print("TOP 5 MOST UNDERSERVED CRISES BY YEAR")
print("=" * 80)

# ISO3 to country name mapping
name_map = {
    "SDN": "Sudan", "MMR": "Myanmar", "AFG": "Afghanistan", "YEM": "Yemen",
    "SYR": "Syria", "COD": "DR Congo", "SSD": "South Sudan", "HTI": "Haiti",
    "VEN": "Venezuela", "COL": "Colombia", "NGA": "Nigeria", "MLI": "Mali",
    "ETH": "Ethiopia", "BGD": "Bangladesh", "PSE": "Palestine", "UKR": "Ukraine"
}
core["country"] = core["iso3"].map(name_map).fillna(core["iso3"])

for year in YEARS:
    df_year = core[core["year"] == year].sort_values("mismatch", ascending=False).head(5)
    print(f"\n{year}:")
    for _, row in df_year.iterrows():
        print(f"  {row['country']:20} | {format_num(row['in_need']):>8} in need | "
              f"{row['need_rate']*100:5.1f}% need rate | ${row['usd_per_in_need']:6.0f}/person | "
              f"mismatch: {row['mismatch']:+.2f}")

In [ ]:
# Visualization: Need Rate vs Resources (2026)
fig, ax = plt.subplots(figsize=(12, 8))

df_2026 = core[core["year"] == 2026].copy()

# Scatter plot
scatter = ax.scatter(
    df_2026["need_rate"] * 100,
    df_2026["usd_per_in_need"],
    s=df_2026["in_need"] / 1e6 * 5,  # Size by population in need
    c=df_2026["mismatch"],
    cmap="RdYlGn_r",
    alpha=0.7,
    edgecolors="black",
    linewidth=0.5
)

# Add country labels for top mismatch countries
for _, row in df_2026.nlargest(7, "mismatch").iterrows():
    ax.annotate(
        row["country"],
        (row["need_rate"] * 100, row["usd_per_in_need"]),
        fontsize=9,
        ha="left",
        va="bottom"
    )

# Add reference lines
median_usd = df_2026["usd_per_in_need"].median()
ax.axhline(median_usd, color="gray", linestyle="--", alpha=0.5, label=f"Median: ${median_usd:.0f}/person")

ax.set_xlabel("Need Rate (% of population in need)", fontsize=12)
ax.set_ylabel("USD Requested per Person in Need", fontsize=12)
ax.set_title("Humanitarian Need vs Resource Allocation (2026)\nLarger bubbles = more people in need | Red = more underserved", fontsize=14)
ax.legend(loc="upper right")

plt.colorbar(scatter, ax=ax, label="Mismatch Score")
plt.tight_layout()
plt.show()

## 4. Mismatch Scoring Model

**Model Approach: Percentile-based Mismatch Index**

We use a **simple, interpretable ranking model** rather than complex ML because:
1. **Transparency** — UN decision-makers need to understand and trust the methodology
2. **Robustness** — Percentile ranks are insensitive to outliers and scale differences
3. **Actionability** — Direct comparison: "Sudan ranks 95th percentile for need but only 15th percentile for resources"

**Mismatch Formula:**
```
mismatch = percentile_rank(need_rate) - percentile_rank(usd_per_person_in_need)
```

**Interpretation:**
- **Positive mismatch** → High need, low resources (underserved / "forgotten")
- **Negative mismatch** → Low need, high resources (potentially over-resourced)
- **Zero mismatch** → Resources proportional to need

In [ ]:
# Add INFORM severity data for validation
inform_agg = (
    inform.groupby(["iso3", "year"], as_index=False)
    .agg({
        "severity_index": "max",
        "crisis_type": lambda x: "|".join(sorted(set(x.dropna()))),
        "primary_driver": lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "Unknown",
        "region": "first"
    })
)

# Use 2025 INFORM data for 2026 (most recent available)
inform_2026 = inform_agg[inform_agg["year"] == 2025].copy()
inform_2026["year"] = 2026
inform_for_join = pd.concat([inform_agg[inform_agg["year"].isin([2024, 2025])], inform_2026], ignore_index=True)

# Merge INFORM with core analysis
core_enriched = core.merge(
    inform_for_join[["iso3", "year", "severity_index", "crisis_type", "primary_driver", "region"]],
    on=["iso3", "year"],
    how="left"
)

# Normalize severity to 0-1 scale
core_enriched["severity_norm"] = core_enriched["severity_index"] / 5.0

# Combined mismatch including severity
core_enriched["mismatch_severity"] = (core_enriched["mismatch"] + core_enriched["severity_norm"]) / 2

print(f"Enriched table: {len(core_enriched)} records")
print(f"INFORM coverage: {core_enriched['severity_index'].notna().sum()} / {len(core_enriched)}")

## 5. Key Findings & Predictions

### Finding 1: Sudan is the most underserved crisis in 2026

Sudan shows the highest mismatch score: **65% of population in need** but only **$85 requested per person** — well below the $120 median. This is particularly concerning given its **severity index of 4.9/5.0** (highest category).

### Finding 2: Conflict-driven crises are systematically underfunded

Analyzing by primary driver shows that conflict-driven crises receive **25% less per-capita** than disaster-driven crises despite similar need rates.

In [ ]:
# Final Forgotten Crisis Index for 2026
df_final = core_enriched[core_enriched["year"] == 2026].sort_values("mismatch_severity", ascending=False)

print("=" * 100)
print("FORGOTTEN CRISIS INDEX 2026 — PRIORITIZED LIST FOR UN DECISION-MAKERS")
print("=" * 100)
print(f"{'Rank':<5} {'Country':<25} {'In Need':>12} {'Need Rate':>10} {'$/Person':>10} {'Severity':>10} {'Mismatch':>10}")
print("-" * 100)

for rank, (_, row) in enumerate(df_final.head(10).iterrows(), 1):
    severity = row['severity_index'] if pd.notna(row['severity_index']) else 0
    print(f"{rank:<5} {row['country']:<25} {format_num(row['in_need']):>12} "
          f"{row['need_rate']*100:>9.1f}% ${row['usd_per_in_need']:>9.0f} "
          f"{severity:>9.1f} {row['mismatch_severity']:>+9.2f}")

print("-" * 100)
print("\nINTERPRETATION: Higher mismatch = more underserved (high need, low resources, high severity)")
print("ACTION: Prioritize top-5 for increased allocation and monitoring")

In [ ]:
# Visualization: Top 10 Forgotten Crises
fig, ax = plt.subplots(figsize=(12, 6))

top10 = df_final.head(10).sort_values("mismatch_severity")

colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(top10)))

bars = ax.barh(top10["country"], top10["mismatch_severity"], color=colors, edgecolor="black")

ax.set_xlabel("Mismatch Score (higher = more underserved)", fontsize=12)
ax.set_title("Top 10 Forgotten Crises (2026)\nBased on Need-Resource Mismatch + Severity", fontsize=14)
ax.axvline(0, color="black", linewidth=0.5)

# Add value labels
for bar, val in zip(bars, top10["mismatch_severity"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f"{val:.2f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Temporal analysis: Track persistent underfunding
persistent = (
    core_enriched.groupby("country")
    .agg({
        "mismatch": "mean",
        "in_need": "mean",
        "usd_per_in_need": "mean",
        "year": "count"
    })
    .rename(columns={"year": "years_present"})
    .query("years_present >= 2")
    .sort_values("mismatch", ascending=False)
)

print("=" * 80)
print("PERSISTENTLY UNDERSERVED COUNTRIES (present in 2+ years)")
print("=" * 80)
print(f"{'Country':<25} {'Avg Mismatch':>12} {'Avg In Need':>15} {'Avg $/Person':>12}")
print("-" * 80)
for country, row in persistent.head(7).iterrows():
    print(f"{country:<25} {row['mismatch']:>+11.2f} {format_num(row['in_need']):>15} ${row['usd_per_in_need']:>11.0f}")

print("-" * 80)
print("\nThese countries show CHRONIC underfunding patterns requiring sustained attention.")

## 6. Conclusions & Recommendations

### Key Conclusions

1. **Systematic mismatch exists** — Countries with the highest need rates do NOT receive proportionally more resources
2. **Sudan stands out** — Highest mismatch score in 2026 with 33.7M people in need
3. **Pattern is persistent** — Same countries (Sudan, Myanmar, Afghanistan) appear underserved across all years
4. **Severity validates findings** — High-severity crises (INFORM 4.5+) are more likely to be underfunded

### Actionable Recommendations for UN Decision-Makers

| Priority | Recommendation | Evidence | Stakeholder |
|----------|---------------|----------|-------------|
| **1** | Increase Sudan allocation by 40% | 65% need rate, $85/person vs $120 median | OCHA, Donors |
| **2** | Establish "forgotten crisis" monitoring | 5 countries consistently underserved | CERF, Pooled Funds |
| **3** | Rebalance regional allocations | Africa receives 25% less per-capita | Donor Coordination |
| **4** | Fast-track conflict-driven crises | Conflict crises systematically underfunded | Early Warning Systems |

### Limitations

1. **revisedRequirements ≠ actual funding** — We measure requested, not received
2. **Admin-level granularity** — Country-level analysis may mask subnational gaps
3. **INFORM lag** — Using 2025 severity for 2026 analysis
4. **Sector gaps not shown** — This analysis is country-level; sector-level gaps exist

### Dashboard Access

Explore the interactive dashboard at: **[datathon-2026.vercel.app](https://datathon-2026.vercel.app)**

The dashboard provides:
- Interactive map of crisis severity and funding gaps
- Drill-down by country and sector
- AI-powered Q&A for exploring the data
- Access to all source datasets and notebooks